# nested-param-group-loop — ex1: manual SGD step via the nested param_groups loop

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nested-param-group-loop`. Running the final beacon cell reports progress against the `Config: nested param-group loop` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: nested param-group loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nested-param-group-loop`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nested-param-group-loop"
DD_SUBTOPIC = "Config: nested param-group loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: nested param-group loop — quick refresher

Every PyTorch optimizer normalizes its params into `self.param_groups: list[dict]`. To touch every parameter tensor you nest two loops:

```python
for group in optimizer.param_groups:
    lr = group['lr']
    for p in group['params']:
        if p.grad is None:
            continue
        p.add_(p.grad, alpha=-lr)
```

**This is the universal optimizer step pattern.** SGD, Adam, RMSprop, AdamW — every reference implementation in `torch/optim/*.py` starts with this nested loop.

**Outer loop = per-group hparam fetch.** `lr`, `weight_decay`, `momentum`, `betas` all live on the group dict. Naive flat-iteration `for p in optimizer.parameters()` doesn't exist on the optimizer and would lose the per-group hparams.

**Skip `p.grad is None` parameters.** When `set_to_none=True` is the zero_grad mode (PyTorch 1.7+ default), un-touched params have `grad=None` and you must not try to read it. The check is one line and prevents crashes on partially-frozen models.

**Inner-loop variable naming.** `p` for the param, `g` for `p.grad`, `lr` for the group LR — these names are conventional across the PyTorch optim source. Match them when you write custom optimizers so reviewers don't have to context-switch.

### Exercise 1 — manual SGD step via the nested param_groups loop

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the canonical `for group in optimizer.param_groups: for p in group['params']: ...` nested loop to manually perform an SGD step that honors per-group learning rates and skips `grad=None` params.
> Keywords: param-groups, sgd-step, manual-optimizer, in-place
> ```

**KCs targeted:** `outer-loop-on-param-groups`, `inner-loop-skip-grad-none`

Implement `ex1_manual_sgd_step(optimizer)`. The no-momentum SGD step, hand-written, using the optimizer's `param_groups` structure.

Algorithm:
```
for group in optimizer.param_groups:
    lr = group['lr']
    for p in group['params']:
        if p.grad is None:
            continue
        p.data.add_(p.grad, alpha=-lr)
```

1. Outer loop iterates over `optimizer.param_groups` (a `list[dict]`).
2. Read `lr = group['lr']` ONCE per group.
3. Inner loop iterates over `group['params']`.
4. If `p.grad is None`, skip — don't try to read it.
5. Otherwise update in place: `p.data.add_(p.grad, alpha=-lr)`.

Do NOT call `optimizer.step()`. You're reimplementing step from scratch.

Output: `None` (in-place mutation of params).

In [ ]:
def ex1_manual_sgd_step(optimizer):
    """Manually perform one SGD step using optimizer.param_groups."""
    raise NotImplementedError()


def _test_ex1():
    # === Single group, simple step ===
    p = t.nn.Parameter(t.tensor([1.0, 2.0, 3.0]))
    opt = t.optim.SGD([p], lr=0.1)
    p.grad = t.tensor([10.0, 20.0, 30.0])
    ex1_manual_sgd_step(opt)
    expected = t.tensor([1.0, 2.0, 3.0]) - 0.1 * t.tensor([10.0, 20.0, 30.0])
    assert t.allclose(p.detach(), expected, atol=1e-6), (
        f'single-group step wrong: got {p.detach()}, expected {expected}'
    )

    # === Two groups with DIFFERENT lr ===
    p1 = t.nn.Parameter(t.zeros(3))
    p2 = t.nn.Parameter(t.zeros(3))
    opt = t.optim.SGD([
        {'params': [p1], 'lr': 0.01},  # slow group
        {'params': [p2], 'lr': 1.0},   # fast group
    ])
    p1.grad = t.ones(3)
    p2.grad = t.ones(3)
    ex1_manual_sgd_step(opt)
    # p1 moved by 0.01 * 1 = 0.01
    # p2 moved by 1.0  * 1 = 1.0
    assert t.allclose(p1.detach(), -0.01 * t.ones(3)), f'slow group wrong: {p1.detach()}'
    assert t.allclose(p2.detach(), -1.0  * t.ones(3)), f'fast group wrong: {p2.detach()}'

    # === grad=None must be SKIPPED, not crash ===
    p_a = t.nn.Parameter(t.tensor([5.0]))
    p_b = t.nn.Parameter(t.tensor([5.0]))
    opt = t.optim.SGD([p_a, p_b], lr=0.1)
    p_a.grad = t.tensor([1.0])
    # p_b.grad intentionally left as None
    ex1_manual_sgd_step(opt)
    assert t.allclose(p_a.detach(), t.tensor([4.9])), 'p_a should have stepped'
    assert t.allclose(p_b.detach(), t.tensor([5.0])), (
        f'p_b had grad=None; should not have moved, got {p_b.detach()}'
    )

    # === In-place: id and storage preserved ===
    p = t.nn.Parameter(t.zeros(4))
    orig_id = id(p)
    orig_ptr = p.data_ptr()
    opt = t.optim.SGD([p], lr=0.1)
    p.grad = t.ones(4)
    ex1_manual_sgd_step(opt)
    assert id(p) == orig_id, 'param object was rebound'
    assert p.data_ptr() == orig_ptr, 'param storage reallocated'

    # === Verify the loop reads lr from the GROUP, not from a constructor capture ===
    # Mutate the group's lr between construction and step.
    p = t.nn.Parameter(t.zeros(3))
    opt = t.optim.SGD([p], lr=0.1)
    opt.param_groups[0]['lr'] = 10.0  # caller bumped lr (e.g. LR schedule)
    p.grad = t.ones(3)
    ex1_manual_sgd_step(opt)
    assert t.allclose(p.detach(), -10.0 * t.ones(3)), (
        f'lr should be read from group dict at step time; got {p.detach()}, expected -10'
    )

    # === Multi-param group: every param in the group gets the group's lr ===
    pa = t.nn.Parameter(t.zeros(2))
    pb = t.nn.Parameter(t.zeros(3))
    opt = t.optim.SGD([{'params': [pa, pb], 'lr': 0.5}])
    pa.grad = t.ones(2)
    pb.grad = t.ones(3) * 2
    ex1_manual_sgd_step(opt)
    assert t.allclose(pa.detach(), -0.5 * t.ones(2)), f'pa wrong: {pa.detach()}'
    assert t.allclose(pb.detach(), -1.0 * t.ones(3)), f'pb wrong: {pb.detach()}'

    # === Function returns None (sentinel for 'in-place side effect') ===
    p = t.nn.Parameter(t.zeros(2))
    opt = t.optim.SGD([p], lr=0.1)
    p.grad = t.ones(2)
    ret = ex1_manual_sgd_step(opt)
    assert ret is None, f'should return None (in-place), got {ret!r}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_manual_sgd_step(optimizer):
    for group in optimizer.param_groups:
        lr = group['lr']
        for p in group['params']:
            if p.grad is None:
                continue
            p.data.add_(p.grad, alpha=-lr)
```

**Why `p.data.add_(p.grad, alpha=-lr)` and not `p -= lr * p.grad`.** Both work, but the fused form is what real PyTorch optim source uses: it avoids the intermediate `lr * p.grad` allocation. `add_(x, alpha=k)` computes `self += k * x` in one CUDA kernel.

**Why read `lr` once per group, not once per param.** If you write `for p in group['params']: lr = group['lr']; ...`, you're paying the dict lookup per-param. For a 100-param model with a 10-step inner loop, that's 1000 lookups instead of 10. Microscopic, but the convention is 'hoist what's group-scoped to the outer loop'.

**`p.data` vs `p` for the in-place mutation.** Inside an optimizer step you usually wrap with `torch.no_grad()` so autograd doesn't track the mutation. PyTorch's optim source uses `p.data.add_` as belt-and-suspenders — it strips the autograd tracking even without the context manager. Either is acceptable in your own code as long as you're consistent.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()